In [14]:
# Importing all required libraries
import pandas as pd
import numpy as np

# Sklearn — pipeline building blocks
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# Joblib — pipeline save and reload karne k liye
import joblib

import os
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")
print(f"Pandas   version : {pd.__version__}")
print(f"Sklearn  version : __version__ not exposed directly — imported successfully")
print(f"Joblib   version : {joblib.__version__}")

All libraries imported successfully!
Pandas   version : 3.0.0
Sklearn  version : __version__ not exposed directly — imported successfully
Joblib   version : 1.5.3


In [15]:

# Creating a mixed dataset

np.random.seed(42)
n = 500  # total samples

data = {
    # Numeric features
    'age'        : np.random.randint(22, 60, n),
    'salary'     : np.random.randint(30000, 120000, n),
    'experience' : np.random.randint(0, 35, n),
    'score'      : np.round(np.random.uniform(40, 100, n), 2),

    # Categorical features
    'department' : np.random.choice(['Engineering', 'HR', 'Finance', 'Marketing', 'Sales'], n),
    'education'  : np.random.choice(['Bachelors', 'Masters', 'PhD', 'Diploma'], n),
    'gender'     : np.random.choice(['Male', 'Female'], n),

    # Target — class imbalance (80% = 0, 20% = 1)
    'promoted'   : np.random.choice([0, 1], size=n, p=[0.80, 0.20])
}

df = pd.DataFrame(data)

# Intentionally add missing values for realism
df.loc[np.random.choice(df.index, 30), 'age']        = np.nan
df.loc[np.random.choice(df.index, 25), 'salary']     = np.nan
df.loc[np.random.choice(df.index, 20), 'department'] = np.nan
df.loc[np.random.choice(df.index, 15), 'education']  = np.nan

# Save as CSV
df.to_csv('employee_data.csv', index=False)

print("Dataset created and saved as 'employee_data.csv'")
print(f"\nShape            : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nClass Distribution (promoted):")
print(df['promoted'].value_counts())
print(f"\nClass Imbalance Ratio : {df['promoted'].value_counts()[0]} : {df['promoted'].value_counts()[1]}")
print(f"\nMissing Values:\n{df.isnull().sum()}")

Dataset created and saved as 'employee_data.csv'

Shape            : 500 rows x 8 columns

Class Distribution (promoted):
promoted
0    396
1    104
Name: count, dtype: int64

Class Imbalance Ratio : 396 : 104

Missing Values:
age           29
salary        24
experience     0
score          0
department    20
education     15
gender         0
promoted       0
dtype: int64


In [16]:

# Load the dataset and inspect shape and dtypes
# This is Stage 1 — RAW DATA

df = pd.read_csv('employee_data.csv')

print("=" * 60)
print("STAGE 1 : RAW DATA")
print("=" * 60)
print(f"\nShape     : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nMissing Values:\n{df.isnull().sum()}")
print(f"\nTarget Distribution:\n{df['promoted'].value_counts()}")

STAGE 1 : RAW DATA

Shape     : 500 rows x 8 columns

Data Types:
age           float64
salary        float64
experience      int64
score         float64
department        str
education         str
gender            str
promoted        int64
dtype: object

First 5 rows:
    age    salary  experience  score department  education  gender  promoted
0  50.0   49065.0          27  47.51         HR    Masters    Male         0
1  36.0  113460.0          30  81.13      Sales    Masters    Male         0
2  29.0       NaN           8  65.82      Sales    Diploma    Male         0
3  42.0   84693.0          28  52.03      Sales  Bachelors  Female         1
4  40.0   56962.0          13  69.50         HR    Diploma    Male         0

Missing Values:
age           29
salary        24
experience     0
score          0
department    20
education     15
gender         0
promoted       0
dtype: int64

Target Distribution:
promoted
0    396
1    104
Name: count, dtype: int64


In [17]:

# Separating features (X) and target (y)
# Also identifying numeric and categorical columns

# Target column
TARGET = 'promoted'

# Feature matrix and target vector
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Identify numeric and categorical columns automatically
numeric_cols     = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print("=" * 60)
print("STAGE 2 : FEATURE SEPARATION")
print("=" * 60)
print(f"\nFeature Matrix X Shape : {X.shape}")
print(f"Target Vector  y Shape : {y.shape}")
print(f"\nNumeric Columns     ({len(numeric_cols)}) : {numeric_cols}")
print(f"Categorical Columns ({len(categorical_cols)}) : {categorical_cols}")
print(f"\nX dtypes:\n{X.dtypes}")

STAGE 2 : FEATURE SEPARATION

Feature Matrix X Shape : (500, 7)
Target Vector  y Shape : (500,)

Numeric Columns     (4) : ['age', 'salary', 'experience', 'score']
Categorical Columns (3) : ['department', 'education', 'gender']

X dtypes:
age           float64
salary        float64
experience      int64
score         float64
department        str
education         str
gender            str
dtype: object


In [18]:

# Train/Test split BEFORE building the pipeline
# This is critical — split first, fit pipeline on train only
# Fitting on full data = data leakage!

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,     # 80% train, 20% test
    random_state = 42,
    stratify     = y         # class imbalance maintain karo dono sets mein
)

print("=" * 60)
print("STAGE 3 : TRAIN / TEST SPLIT (No Leakage)")
print("=" * 60)
print(f"\nX_train Shape : {X_train.shape}")
print(f"X_test  Shape : {X_test.shape}")
print(f"y_train Shape : {y_train.shape}")
print(f"y_test  Shape : {y_test.shape}")

print(f"\nTrain Target Distribution:")
print(y_train.value_counts())

print(f"\nTest Target Distribution:")
print(y_test.value_counts())

print(f"\nX_train dtypes:\n{X_train.dtypes}")

print("""
NOTE — No Leakage Strategy:
   Pipeline sirf X_train par fit hoga (pipeline.fit)
   X_test par sirf transform hoga (pipeline.transform)
   Test data ki koi bhi information training mein nahi jaayegi
""")

STAGE 3 : TRAIN / TEST SPLIT (No Leakage)

X_train Shape : (400, 7)
X_test  Shape : (100, 7)
y_train Shape : (400,)
y_test  Shape : (100,)

Train Target Distribution:
promoted
0    317
1     83
Name: count, dtype: int64

Test Target Distribution:
promoted
0    79
1    21
Name: count, dtype: int64

X_train dtypes:
age           float64
salary        float64
experience      int64
score         float64
department        str
education         str
gender            str
dtype: object

NOTE — No Leakage Strategy:
   Pipeline sirf X_train par fit hoga (pipeline.fit)
   X_test par sirf transform hoga (pipeline.transform)
   Test data ki koi bhi information training mein nahi jaayegi



In [19]:

# Numeric sub-pipeline
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # median se missing fill karo
    ('scaler',  StandardScaler())                   # standardize karo
])

# Categorical sub-pipeline
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # mode se fill karo
    ('encoder', OneHotEncoder(handle_unknown='ignore',
                              sparse_output=False))        # one hot encode karo
])

# ColumnTransformer — dono pipelines combine karo
preprocessor = ColumnTransformer(transformers=[
    ('numeric',      numeric_pipeline,      numeric_cols),
    ('categorical',  categorical_pipeline,  categorical_cols)
])

# Final Pipeline
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

print("=" * 60)
print("STAGE 4 : PIPELINE STRUCTURE")
print("=" * 60)
print("\nFull Pipeline Steps:")
print(full_pipeline)
print(f"\nNumeric columns being processed    : {numeric_cols}")
print(f"Categorical columns being processed : {categorical_cols}")

STAGE 4 : PIPELINE STRUCTURE

Full Pipeline Steps:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'salary',
                                                   'experience', 'score']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                    

In [20]:

# Fit the pipeline on TRAINING data only
# No leakage — test data is never seen here

print("=" * 60)
print("STAGE 5 : PIPELINE FITTING (Train Only)")
print("=" * 60)

print(f"\nX_train shape BEFORE fit : {X_train.shape}")
print(f"X_train dtypes BEFORE fit:\n{X_train.dtypes}")

# Fit pipeline on training data only
full_pipeline.fit(X_train, y_train)

print("\nPipeline fitted on X_train successfully!")
print("(Mean, Std, Categories — sab X_train se seekha gaya)")

STAGE 5 : PIPELINE FITTING (Train Only)

X_train shape BEFORE fit : (400, 7)
X_train dtypes BEFORE fit:
age           float64
salary        float64
experience      int64
score         float64
department        str
education         str
gender            str
dtype: object

Pipeline fitted on X_train successfully!
(Mean, Std, Categories — sab X_train se seekha gaya)


In [21]:

# Transform both train and test data using fitted pipeline

# Transform train data
X_train_transformed = full_pipeline.transform(X_train)

# Transform test data (same pipeline — no refit)
X_test_transformed  = full_pipeline.transform(X_test)

# Get feature names after OneHotEncoding
ohe_feature_names = (
    full_pipeline
    .named_steps['preprocessor']
    .named_transformers_['categorical']
    .named_steps['encoder']
    .get_feature_names_out(categorical_cols)
    .tolist()
)

all_feature_names = numeric_cols + ohe_feature_names

print("=" * 60)
print("STAGE 6 : DATA AFTER TRANSFORMATION")
print("=" * 60)

print(f"\nX_train BEFORE transform : shape = {X_train.shape}, dtype = {X_train.dtypes.unique()}")
print(f"X_train AFTER  transform : shape = {X_train_transformed.shape}, dtype = {X_train_transformed.dtype}")

print(f"\nX_test  BEFORE transform : shape = {X_test.shape},  dtype = {X_test.dtypes.unique()}")
print(f"X_test  AFTER  transform : shape = {X_test_transformed.shape},  dtype = {X_test_transformed.dtype}")

print(f"\nTotal Features After Encoding : {X_train_transformed.shape[1]}")
print(f"\nFeature Names After Pipeline:")
for i, name in enumerate(all_feature_names):
    print(f"   [{i:02d}] {name}")

# Show sample of transformed data
transformed_df = pd.DataFrame(X_train_transformed, columns=all_feature_names)
print(f"\nSample of Transformed Training Data (first 3 rows):")
print(transformed_df.head(3).round(4))

STAGE 6 : DATA AFTER TRANSFORMATION

X_train BEFORE transform : shape = (400, 7), dtype = [dtype('float64') dtype('int64')
 <StringDtype(storage='python', na_value=nan)>]
X_train AFTER  transform : shape = (400, 15), dtype = float64

X_test  BEFORE transform : shape = (100, 7),  dtype = [dtype('float64') dtype('int64')
 <StringDtype(storage='python', na_value=nan)>]
X_test  AFTER  transform : shape = (100, 15),  dtype = float64

Total Features After Encoding : 15

Feature Names After Pipeline:
   [00] age
   [01] salary
   [02] experience
   [03] score
   [04] department_Engineering
   [05] department_Finance
   [06] department_HR
   [07] department_Marketing
   [08] department_Sales
   [09] education_Bachelors
   [10] education_Diploma
   [11] education_Masters
   [12] education_PhD
   [13] gender_Female
   [14] gender_Male

Sample of Transformed Training Data (first 3 rows):
      age  salary  experience   score  department_Engineering  \
0  0.6847 -1.3895      1.3045 -0.1268        

In [22]:

# Save the fitted pipeline using joblib
# Joblib efficiently saves sklearn objects to disk

pipeline_path = 'fitted_pipeline.joblib'
joblib.dump(full_pipeline, pipeline_path)

file_size = os.path.getsize(pipeline_path) / 1024  # KB mein

print("=" * 60)
print("STAGE 7 : PIPELINE SAVED")
print("=" * 60)
print(f"\nPipeline saved to : '{pipeline_path}'")
print(f"File Size         : {file_size:.2f} KB")
print("\nPipeline is now ready to be reloaded and reused!")

STAGE 7 : PIPELINE SAVED

Pipeline saved to : 'fitted_pipeline.joblib'
File Size         : 3.98 KB

Pipeline is now ready to be reloaded and reused!


In [23]:

# Reload the pipeline from disk
# This simulates production use — load once, use many times

reloaded_pipeline = joblib.load(pipeline_path)

print("=" * 60)
print("STAGE 8 : PIPELINE RELOADED FROM DISK")
print("=" * 60)
print(f"\nPipeline reloaded from : '{pipeline_path}'")
print("\nReloaded Pipeline Structure:")
print(reloaded_pipeline)
print("\nPipeline reloaded successfully!")

STAGE 8 : PIPELINE RELOADED FROM DISK

Pipeline reloaded from : 'fitted_pipeline.joblib'

Reloaded Pipeline Structure:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'salary',
                                                   'experience', 'score']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                              

In [24]:

# Creating brand new raw samples — simulating real-world
# incoming data that needs to be preprocessed
# Using reloaded pipeline to transform — end to end demo
# 5 new raw samples — same format as original data
new_raw_samples = pd.DataFrame({
    'age'        : [28, None, 45, 33, 52],         # one missing
    'salary'     : [55000, 72000, None, 48000, 95000],  # one missing
    'experience' : [5, 10, 20, 7, 30],
    'score'      : [78.5, 85.0, 91.2, 65.3, 88.7],
    'department' : ['Engineering', None, 'Finance', 'HR', 'Marketing'],  # one missing
    'education'  : ['Masters', 'Bachelors', None, 'PhD', 'Diploma'],     # one missing
    'gender'     : ['Male', 'Female', 'Male', 'Female', 'Male']
})

print("=" * 60)
print("STAGE 9 : END TO END — NEW RAW SAMPLES")
print("=" * 60)

print(f"\nNew Raw Samples Shape  : {new_raw_samples.shape}")
print(f"New Raw Samples dtypes :\n{new_raw_samples.dtypes}")
print(f"\nRaw Samples (before transform):")
print(new_raw_samples)

# Transform using RELOADED pipeline — no refit needed
new_transformed = reloaded_pipeline.transform(new_raw_samples)

print(f"\nAfter Transform Shape  : {new_transformed.shape}")
print(f"After Transform dtype  : {new_transformed.dtype}")

# Show as readable DataFrame
new_transformed_df = pd.DataFrame(new_transformed, columns=all_feature_names)
print(f"\nTransformed New Samples (first 5 features shown):")
print(new_transformed_df.iloc[:, :5].round(4))

print("\nEnd to end pipeline demo successful!")
print("Raw data  →  Imputed  →  Scaled/Encoded  →  Model-ready")

STAGE 9 : END TO END — NEW RAW SAMPLES

New Raw Samples Shape  : (5, 7)
New Raw Samples dtypes :
age           float64
salary        float64
experience      int64
score         float64
department        str
education         str
gender            str
dtype: object

Raw Samples (before transform):
    age   salary  experience  score   department  education  gender
0  28.0  55000.0           5   78.5  Engineering    Masters    Male
1   NaN  72000.0          10   85.0          NaN  Bachelors  Female
2  45.0      NaN          20   91.2      Finance        NaN    Male
3  33.0  48000.0           7   65.3           HR        PhD  Female
4  52.0  95000.0          30   88.7    Marketing    Diploma    Male

After Transform Shape  : (5, 15)
After Transform dtype  : float64

Transformed New Samples (first 5 features shown):
      age  salary  experience   score  department_Engineering
0 -1.2716 -0.6219     -1.2801  0.5167                     1.0
1  0.1258  0.0245     -0.7831  0.8877               

In [25]:

# Complete pipeline summary — all stages recap

print("=" * 65)
print("          PREPROCESSING PIPELINE — FULL SUMMARY")
print("=" * 65)

print(f"""
DATASET
   File              : employee_data.csv
   Raw Shape         : {df.shape[0]} rows x {df.shape[1]} columns
   Numeric Features  : {numeric_cols}
   Categorical Feats : {categorical_cols}
   Target            : promoted (class imbalance 80/20)

TRAIN / TEST SPLIT
   Train Size        : {X_train.shape[0]} rows (80%)
   Test  Size        : {X_test.shape[0]} rows  (20%)
   Stratified        : Yes (class ratio maintained)

PIPELINE STEPS
   Numeric           : SimpleImputer(median) → StandardScaler
   Categorical       : SimpleImputer(mode)   → OneHotEncoder

TRANSFORMATION RESULTS
   Before Transform  : {X_train.shape[1]} columns (mixed types)
   After  Transform  : {X_train_transformed.shape[1]} columns (all float64)
   New features from OHE: {len(ohe_feature_names)}

NO LEAKAGE CONFIRMATION
   Pipeline fit on   : X_train only
   Pipeline transform: X_train + X_test separately
   Test data seen    : Never during fit

PIPELINE PERSISTENCE
   Saved to          : fitted_pipeline.joblib
   Reloaded          : Successfully
   New batch test    : {new_raw_samples.shape[0]} samples → {new_transformed.shape[1]} features

STATUS : All stages completed successfully.
""")
print("=" * 65)

          PREPROCESSING PIPELINE — FULL SUMMARY

DATASET
   File              : employee_data.csv
   Raw Shape         : 500 rows x 8 columns
   Numeric Features  : ['age', 'salary', 'experience', 'score']
   Categorical Feats : ['department', 'education', 'gender']
   Target            : promoted (class imbalance 80/20)

TRAIN / TEST SPLIT
   Train Size        : 400 rows (80%)
   Test  Size        : 100 rows  (20%)
   Stratified        : Yes (class ratio maintained)

PIPELINE STEPS
   Numeric           : SimpleImputer(median) → StandardScaler
   Categorical       : SimpleImputer(mode)   → OneHotEncoder

TRANSFORMATION RESULTS
   Before Transform  : 7 columns (mixed types)
   After  Transform  : 15 columns (all float64)
   New features from OHE: 11

NO LEAKAGE CONFIRMATION
   Pipeline fit on   : X_train only
   Pipeline transform: X_train + X_test separately
   Test data seen    : Never during fit

PIPELINE PERSISTENCE
   Saved to          : fitted_pipeline.joblib
   Reloaded          